
# Production Image Optimization + XMP + Local Vision Metadata

**Status:** Production prototype v0.1  
**Basis:** `spec.md` + calibrated image rules from `image_optimization_calibration.v04.ipynb`

This notebook processes Docling-extracted raster assets end-to-end:

1. Parse Docling JSON + Markdown.
2. Resolve every referenced picture.
3. Downsample to a hard maximum of **144 ppi** based on Docling/PDF geometry.
4. Deterministically classify and optimize to **JPEG or PNG only**.
5. Optionally call a **local vision model via LM Studio** for semantic metadata.
6. Construct XMP using Dublin Core / IPTC fields plus a small `docrag` namespace.
7. Embed XMP directly into final JPEG/PNG files.
8. Re-open and validate XMP after writing.
9. Rewrite Markdown image paths and alt text.
10. Write JSON/CSV audit manifests.

Current reference basis:

- W3C PNG Third Edition: https://www.w3.org/TR/png-3/
- IPTC Photo Metadata 2025.1: https://www.iptc.org/std/photometadata/specification/IPTC-PhotoMetadata-2025.1.html
- LM Studio OpenAI compatibility: https://lmstudio.ai/docs/developer/openai-compat
- LM Studio Structured Output: https://lmstudio.ai/docs/developer/openai-compat/structured-output
- ExifTool XMP tags: https://exiftool.org/TagNames/XMP.html
- ExifTool PNG tags: https://exiftool.org/TagNames/PNG.html


In [ ]:

from __future__ import annotations

import base64
import binascii
import hashlib
import io
import json
import os
import re
import struct
import urllib.request
import subprocess
import tempfile
import shutil
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from xml.sax.saxutils import escape

import numpy as np
import pandas as pd
from PIL import Image
import cv2
from skimage.metrics import structural_similarity as ssim
from skimage.color import rgb2lab, deltaE_ciede2000

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)


## 00 — Configuration

In [ ]:

from pathlib import Path
import sys

# Projekt-Root wie in 01/02 robust bestimmen.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        p for p in (cwd, *cwd.parents)
        if (p / "data" / "processed").is_dir()
        and (p / "data" / "raw").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Projekt-Root nicht gefunden: data/raw und data/processed fehlen."
    )

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"

# --- Anzupassen
BUCH = None
LMS = "http://192.168.178.27:1234/v1"
MODELL = "google/gemma-4-12b"

# Optional: explizites Markdown setzen. None = <BUCH>.md.
MARKDOWN_OVERRIDE = None

# Dokument automatisch aus data/raw bestimmen – analog zu Notebook 01/02.
gefunden = (
    sorted(
        p.stem
        for p in RAW.iterdir()
        if p.is_file() and p.suffix.lower() == ".pdf"
    )
    if RAW.exists()
    else []
)

if BUCH is None:
    if len(gefunden) == 1:
        BUCH = gefunden[0]
        print(f"BUCH automatisch auf {BUCH!r} gesetzt.")
    elif gefunden:
        BUCH = gefunden[0]
        print(
            f"! {len(gefunden)} PDFs gefunden – BUCH oben ggf. von Hand setzen. "
            f"Vorläufig: {BUCH!r}"
        )
    else:
        raise RuntimeError("Kein PDF unter data/raw gefunden.")

DOCUMENT_DIR = PROCESSED / BUCH
MARKDOWN_FILE = (
    Path(MARKDOWN_OVERRIDE)
    if MARKDOWN_OVERRIDE is not None
    else DOCUMENT_DIR / f"{BUCH}.md"
)
DOCLING_JSON = DOCUMENT_DIR / f"{BUCH}.json"
ARTIFACT_DIR = DOCUMENT_DIR / f"{BUCH}_artifacts"

# In-place bedeutet hier:
# - Rasterdateien werden im bestehenden <BUCH>_artifacts-Ordner ersetzt.
# - Markdown und Docling-JSON werden nach erfolgreicher Validierung aktualisiert.
# - Es entsteht KEIN zusätzlicher _optimized_document-Ordner.
IN_PLACE = True

# Kleines Audit-Artefakt; keine Bildkopie.
MANIFEST_JSON = DOCUMENT_DIR / f"{BUCH}.image-optimization.json"
MANIFEST_CSV = DOCUMENT_DIR / f"{BUCH}.image-optimization.csv"

# ---------------------------------------------------------------------------
# Hard project policy
# ---------------------------------------------------------------------------
MAX_PPI = 144
ALLOW_UPSCALE = False
MIN_SAVING_RATIO = 0.10

JPEG_QUALITIES = [85, 75, 65, 55, 45]
PNG_PALETTE_SIZES = [2, 4, 8, 16, 32, 64, 128, 256]
COLOR_GRAPHIC_MIN_PALETTE_COLORS = 16

ENABLE_PNGQUANT = True
PNGQUANT_BINARY = shutil.which("pngquant")
PNGQUANT_SPEED = 1
PNGQUANT_QUALITY_PROFILES = [(85, 100), (75, 95)]

QUALITY = {
    "photo_ssim_min": 0.88,
    "mixed_ssim_min": 0.90,
    "graphic_ssim_min": 0.93,
    "edge_f1_min": 0.80,
    "bilevel_edge_f1_min": 0.86,
    "component_retention_min": 0.80,
    "color_deltae_mean_max": 6.0,
    "color_deltae_p95_max": 16.0,
    "preferred_graphic_ssim_min": 0.96,
    "preferred_edge_f1_min": 0.90,
    "preferred_component_retention_min": 0.90,
    "preferred_color_deltae_mean_max": 4.0,
    "preferred_color_deltae_p95_max": 10.0,
}

CLASSIFIER = {
    "grayscale_distance_max": 0.020,
    "bilevel_extreme_fraction_min": 0.90,
    "bilevel_midtone_fraction_max": 0.08,
    "graphic_flat_area_min": 0.30,
    "graphic_edge_density_min": 0.12,
    "graphic_extreme_fraction_min": 0.45,
    "graphic_low_entropy_max": 0.78,
}

FEATURE_MAX_SIDE = 768
METADATA_LANGUAGE = "de"
DOCRAG_NAMESPACE_URI = "urn:docrag:metadata:1.0"
DOCRAG_SCHEMA_VERSION = "1.0"

ENABLE_LLM = True
LM_STUDIO_BASE_URL = LMS
LM_STUDIO_MODEL = MODELL
LM_STUDIO_TIMEOUT_SECONDS = 180
LM_STUDIO_TEMPERATURE = 0.1
LM_STUDIO_MAX_TOKENS = 1200
MAX_KEYWORDS = 12
MAX_ALT_TEXT_CHARS = 250

for label, path in [
    ("Projekt-Root", PROJECT_ROOT),
    ("Dokumentordner", DOCUMENT_DIR),
    ("Markdown", MARKDOWN_FILE),
    ("Docling JSON", DOCLING_JSON),
    ("Artifacts", ARTIFACT_DIR),
]:
    print(f"{label:15s}: {path} -> {'vorhanden' if path.exists() else 'FEHLT'}")

if not MARKDOWN_FILE.exists():
    raise FileNotFoundError(MARKDOWN_FILE)
if not DOCLING_JSON.exists():
    raise FileNotFoundError(DOCLING_JSON)
if not ARTIFACT_DIR.is_dir():
    raise FileNotFoundError(ARTIFACT_DIR)


## 01 — Parse Docling, Markdown and contextual metadata

In [ ]:

def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def source_document_name(doc: dict[str, Any]) -> str:
    return doc.get("origin", {}).get("filename") or doc.get("name") or DOCLING_JSON.stem

def deref_text(doc: dict[str, Any], ref: str | None) -> str | None:
    if not ref or not ref.startswith("#/texts/"):
        return None
    try:
        idx = int(ref.rsplit("/", 1)[-1])
        item = doc["texts"][idx]
        return item.get("text") or item.get("orig")
    except Exception:
        return None

def picture_caption(doc: dict[str, Any], picture: dict[str, Any]) -> str | None:
    vals = []
    for item in picture.get("captions", []):
        if isinstance(item, dict):
            val = deref_text(doc, item.get("$ref"))
            if val and val.strip():
                vals.append(val.strip())
    return " ".join(vals) if vals else None

def nearest_section_heading(doc: dict[str, Any], page_no: int, picture_top: float) -> str | None:
    candidates = []
    for item in doc.get("texts", []):
        if item.get("label") != "section_header":
            continue
        for prov in item.get("prov", []):
            if prov.get("page_no") == page_no:
                top = float(prov.get("bbox", {}).get("t", 0))
                if top <= picture_top:
                    candidates.append((top, item.get("text") or item.get("orig") or ""))
    return max(candidates, key=lambda x: x[0])[1].strip() if candidates else None

def nearby_text(doc: dict[str, Any], page_no: int, bbox: dict[str, Any], max_chars: int = 1800) -> str:
    pt, pb = float(bbox.get("t", 0)), float(bbox.get("b", 0))
    candidates = []
    for item in doc.get("texts", []):
        text = (item.get("text") or item.get("orig") or "").strip()
        if not text:
            continue
        for prov in item.get("prov", []):
            if prov.get("page_no") != page_no:
                continue
            tb = prov.get("bbox", {})
            tt, tbot = float(tb.get("t", 0)), float(tb.get("b", 0))
            dist = max(0.0, max(tt, pt) - min(tbot, pb))
            candidates.append((dist, tt, text))
            break
    chunks, used = [], 0
    for _, _, text in sorted(candidates, key=lambda x: (x[0], x[1])):
        if used >= max_chars:
            break
        piece = text[:max_chars-used]
        chunks.append(piece)
        used += len(piece) + 2
    return "\n\n".join(chunks)

doc = load_json(DOCLING_JSON)
md_text = MARKDOWN_FILE.read_text(encoding="utf-8")
print("Pictures:", len(doc.get("pictures", [])))


## 02 — Asset manifest and 144 ppi geometry

In [ ]:

def resolve_asset(uri: str) -> Path | None:
    """
    Resolve only inside the current processed document.
    Avoids accidental basename matches in other books.
    """
    uri_path = Path(uri)

    candidates = [
        DOCUMENT_DIR / uri_path,
        ARTIFACT_DIR / uri_path.name,
    ]
    for p in candidates:
        if p.exists() and p.is_file():
            return p.resolve()

    return None

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

def build_asset_manifest(doc: dict[str, Any]) -> pd.DataFrame:
    rows = []
    for picture in doc.get("pictures", []):
        prov = (picture.get("prov") or [{}])[0]
        bbox = prov.get("bbox") or {}
        image = picture.get("image") or {}
        size = image.get("size") or {}
        uri = image.get("uri")
        resolved = resolve_asset(uri) if uri else None

        l, t = float(bbox.get("l", 0)), float(bbox.get("t", 0))
        r, b = float(bbox.get("r", 0)), float(bbox.get("b", 0))
        bbox_w, bbox_h = abs(r-l), abs(b-t)

        max_w = max(1, round(bbox_w / 72 * MAX_PPI))
        max_h = max(1, round(bbox_h / 72 * MAX_PPI))

        source_w = int(round(float(size.get("width", 0) or 0)))
        source_h = int(round(float(size.get("height", 0) or 0)))
        scale = min(max_w/source_w, max_h/source_h) if source_w and source_h else 1.0
        if not ALLOW_UPSCALE:
            scale = min(1.0, scale)

        page_no = prov.get("page_no")
        rows.append({
            "picture_ref": picture.get("self_ref"),
            "page_no": page_no,
            "bbox_l": l, "bbox_t": t, "bbox_r": r, "bbox_b": b,
            "bbox_coord_origin": bbox.get("coord_origin"),
            "source_uri": uri,
            "source_declared_dpi": image.get("dpi"),
            "source_declared_width": source_w,
            "source_declared_height": source_h,
            "target_width": max(1, round(source_w * scale)),
            "target_height": max(1, round(source_h * scale)),
            "resolved_path": str(resolved) if resolved else None,
            "exists": bool(resolved),
            "source_caption": picture_caption(doc, picture),
            "section_heading": nearest_section_heading(doc, page_no, t) if page_no else None,
            "nearby_text": nearby_text(doc, page_no, bbox) if page_no else "",
        })
    return pd.DataFrame(rows)

assets = build_asset_manifest(doc)
display(assets[["picture_ref","page_no","source_uri","target_width","target_height","exists"]])


## 03 — Calibrated deterministic classification

In [ ]:

def analysis_thumbnail(img: Image.Image) -> Image.Image:
    out = img.copy()
    out.thumbnail((FEATURE_MAX_SIDE, FEATURE_MAX_SIDE), Image.Resampling.LANCZOS)
    return out

def pil_to_rgb_array(img: Image.Image) -> np.ndarray:
    if "A" in img.getbands():
        bg = Image.new("RGBA", img.size, (255,255,255,255))
        bg.alpha_composite(img.convert("RGBA"))
        return np.asarray(bg.convert("RGB"))
    return np.asarray(img.convert("RGB"))

def normalized_entropy(gray: np.ndarray) -> float:
    hist = cv2.calcHist([gray],[0],None,[256],[0,256]).ravel()
    p = hist / max(hist.sum(), 1)
    p = p[p > 0]
    return float(-(p*np.log2(p)).sum()/8.0)

def technical_features(path: Path) -> dict[str, Any]:
    with Image.open(path) as src:
        src.load()
        has_alpha = "A" in src.getbands()
        alpha_fraction = float(np.mean(np.asarray(src.getchannel("A")) < 255)) if has_alpha else 0.0
        small = analysis_thumbnail(src)
        rgb = pil_to_rgb_array(small).astype(np.uint8)
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

        rgbf = rgb.astype(np.float32)/255.0
        chdiff = np.maximum.reduce([
            np.abs(rgbf[...,0]-rgbf[...,1]),
            np.abs(rgbf[...,0]-rgbf[...,2]),
            np.abs(rgbf[...,1]-rgbf[...,2]),
        ])

        med = float(np.median(gray))
        edges = cv2.Canny(gray, int(max(0,0.66*med)), int(min(255,1.33*med)))

        g = gray.astype(np.float32)
        mean = cv2.GaussianBlur(g,(0,0),2.0)
        mean2 = cv2.GaussianBlur(g*g,(0,0),2.0)
        var = np.maximum(mean2-mean*mean,0)

        return {
            "actual_width": src.width,
            "actual_height": src.height,
            "actual_mode": src.mode,
            "has_alpha": has_alpha,
            "alpha_fraction_nonopaque": alpha_fraction,
            "grayscale_distance": float(np.mean(chdiff)),
            "luminance_entropy": normalized_entropy(gray),
            "edge_density": float(np.mean(edges > 0)),
            "flat_area_fraction": float(np.mean(var < 9.0)),
            "extreme_fraction": float(np.mean(gray <= 24) + np.mean(gray >= 231)),
            "midtone_fraction": float(np.mean((gray > 48) & (gray < 207))),
        }

def classify_image(f: dict[str, Any]) -> tuple[str, list[str]]:
    if f["has_alpha"] and f["alpha_fraction_nonopaque"] > 0:
        return "ALPHA_GRAPHIC", ["non-opaque alpha"]

    grayscale = f["grayscale_distance"] <= CLASSIFIER["grayscale_distance_max"]
    bilevel = (
        grayscale
        and f["extreme_fraction"] >= CLASSIFIER["bilevel_extreme_fraction_min"]
        and f["midtone_fraction"] <= CLASSIFIER["bilevel_midtone_fraction_max"]
    )
    if bilevel:
        return "LINE_ART_BILEVEL", ["near-bilevel grayscale profile"]

    graphic = (
        (f["flat_area_fraction"] >= CLASSIFIER["graphic_flat_area_min"] and f["edge_density"] >= CLASSIFIER["graphic_edge_density_min"])
        or (f["flat_area_fraction"] >= CLASSIFIER["graphic_flat_area_min"] and f["extreme_fraction"] >= CLASSIFIER["graphic_extreme_fraction_min"])
        or (f["luminance_entropy"] <= CLASSIFIER["graphic_low_entropy_max"] and f["extreme_fraction"] >= CLASSIFIER["graphic_extreme_fraction_min"])
        or (f["edge_density"] >= CLASSIFIER["graphic_edge_density_min"] and f["extreme_fraction"] >= CLASSIFIER["graphic_extreme_fraction_min"])
    )
    if graphic:
        return ("LINE_ART_GRAYSCALE" if grayscale else "SCREENSHOT_OR_TEXT_GRAPHIC"), ["graphic/text-like profile"]

    if grayscale:
        return "PHOTO_GRAYSCALE", ["grayscale continuous-tone fallback"]

    if (
        f["flat_area_fraction"] < CLASSIFIER["graphic_flat_area_min"]
        and f["edge_density"] < CLASSIFIER["graphic_edge_density_min"]
        and f["luminance_entropy"] > CLASSIFIER["graphic_low_entropy_max"]
    ):
        return "PHOTO_COLOR", ["continuous-tone photographic profile"]

    return "MIXED_CONTENT", ["ambiguous technical profile"]


## 04 — Candidate generation and quality gates

In [ ]:

@dataclass
class Candidate:
    candidate_id: str
    format: str
    params: dict[str, Any]
    data: bytes
    image: Image.Image

    @property
    def nbytes(self):
        return len(self.data)

def resize_reference(path: Path, tw: int, th: int) -> Image.Image:
    with Image.open(path) as img:
        img.load()
        scale = min(tw / img.width, th / img.height, 1.0)
        size = (max(1, round(img.width * scale)), max(1, round(img.height * scale)))
        return img.copy() if img.size == size else img.resize(size, Image.Resampling.LANCZOS)

def decode(data: bytes) -> Image.Image:
    im = Image.open(io.BytesIO(data))
    im.load()
    return im

def encode_jpeg(img: Image.Image, quality: int, grayscale=False) -> Candidate:
    out = img.convert("L" if grayscale else "RGB")
    bio = io.BytesIO()
    out.save(
        bio,
        format="JPEG",
        quality=quality,
        optimize=True,
        progressive=True,
        subsampling=0 if grayscale else 2,
    )
    data = bio.getvalue()
    return Candidate(
        f"jpeg_q{quality}_{'gray' if grayscale else 'rgb'}",
        "JPEG",
        {"quality": quality, "grayscale": grayscale},
        data,
        decode(data),
    )

def encode_png_truecolor(img: Image.Image, grayscale=False) -> Candidate:
    out = img.convert("L" if grayscale else ("RGBA" if "A" in img.getbands() else "RGB"))
    bio = io.BytesIO()
    out.save(bio, format="PNG", optimize=True)
    data = bio.getvalue()
    return Candidate(
        f"png_{'gray' if grayscale else 'truecolor'}",
        "PNG",
        {"grayscale": grayscale, "encoder": "pillow"},
        data,
        decode(data),
    )

def encode_png_palette(img: Image.Image, colors: int, dither=False) -> Candidate:
    q = img.convert("RGB").quantize(
        colors=colors,
        method=Image.Quantize.MEDIANCUT,
        dither=Image.Dither.FLOYDSTEINBERG if dither else Image.Dither.NONE,
    )
    bio = io.BytesIO()
    q.save(bio, format="PNG", optimize=True)
    data = bio.getvalue()
    return Candidate(
        f"png_palette_{colors}_{'dither' if dither else 'nodither'}",
        "PNG",
        {"palette_colors": colors, "dither": dither, "encoder": "pillow"},
        data,
        decode(data),
    )

def encode_pngquant(
    img: Image.Image,
    colors: int,
    quality_min: int,
    quality_max: int,
) -> Candidate | None:
    if not (ENABLE_PNGQUANT and PNGQUANT_BINARY):
        return None

    with tempfile.TemporaryDirectory() as td:
        td = Path(td)
        input_path = td / "input.png"
        output_path = td / "output.png"

        img.convert("RGB").save(input_path, format="PNG", optimize=True)

        cmd = [
            PNGQUANT_BINARY,
            str(colors),
            f"--quality={quality_min}-{quality_max}",
            f"--speed={PNGQUANT_SPEED}",
            "--force",
            "--output", str(output_path),
            str(input_path),
        ]

        proc = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=False,
        )

        if proc.returncode != 0 or not output_path.exists():
            return None

        data = output_path.read_bytes()
        return Candidate(
            f"pngquant_{colors}_q{quality_min}-{quality_max}",
            "PNG",
            {
                "palette_colors": colors,
                "quality_min": quality_min,
                "quality_max": quality_max,
                "encoder": "pngquant",
            },
            data,
            decode(data),
        )

def encode_png_bilevel(img: Image.Image) -> Candidate:
    gray = np.asarray(img.convert("L"))
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    out = Image.fromarray(bw, mode="L").convert("1")
    bio = io.BytesIO()
    out.save(bio, format="PNG", optimize=True)
    data = bio.getvalue()
    return Candidate(
        "png_bilevel_otsu",
        "PNG",
        {"threshold": "otsu"},
        data,
        decode(data),
    )

def palette_candidates(
    ref: Image.Image,
    *,
    min_colors: int,
    include_pngquant: bool = True,
) -> list[Candidate]:
    out = []
    sizes = [n for n in PNG_PALETTE_SIZES if n >= min_colors]

    # Pillow baseline candidates.
    out += [encode_png_palette(ref, n, dither=False) for n in sizes]

    # Dither only from 32 colors upward.
    out += [
        encode_png_palette(ref, n, dither=True)
        for n in sizes
        if n >= 32
    ]

    # Optional perceptual pngquant/libimagequant candidates.
    if include_pngquant and ENABLE_PNGQUANT and PNGQUANT_BINARY:
        for n in sizes:
            for qmin, qmax in PNGQUANT_QUALITY_PROFILES:
                cand = encode_pngquant(ref, n, qmin, qmax)
                if cand is not None:
                    out.append(cand)

    return out

def candidate_family(ref: Image.Image, cls: str) -> list[Candidate]:
    if cls == "PHOTO_COLOR":
        return [encode_jpeg(ref, q) for q in JPEG_QUALITIES] + [encode_png_truecolor(ref)]

    if cls == "PHOTO_GRAYSCALE":
        return [encode_jpeg(ref, q, True) for q in JPEG_QUALITIES] + [encode_png_truecolor(ref, True)]

    if cls == "LINE_ART_BILEVEL":
        return (
            [encode_png_bilevel(ref), encode_png_truecolor(ref, True)]
            + palette_candidates(ref, min_colors=2)
        )

    if cls == "LINE_ART_GRAYSCALE":
        return [encode_png_truecolor(ref, True)] + palette_candidates(ref, min_colors=2)

    if cls in {"SCREENSHOT_OR_TEXT_GRAPHIC", "ALPHA_GRAPHIC"}:
        # Hard policy: colored technical graphics never use 2/4/8-color palettes.
        return (
            [encode_png_truecolor(ref)]
            + palette_candidates(
                ref,
                min_colors=COLOR_GRAPHIC_MIN_PALETTE_COLORS,
            )
        )

    # Mixed content: JPEG competes with PNG, but colored palette PNGs obey the same floor.
    return (
        [encode_jpeg(ref, q) for q in JPEG_QUALITIES]
        + [encode_png_truecolor(ref)]
        + palette_candidates(
            ref,
            min_colors=COLOR_GRAPHIC_MIN_PALETTE_COLORS,
        )
    )


In [ ]:

def gray_array(img):
    return np.asarray(img.convert("L"), dtype=np.uint8)

def rgb_float(img, size=None):
    im = img.convert("RGB")
    if size and im.size != size:
        im = im.resize(size, Image.Resampling.BILINEAR)
    return np.asarray(im, dtype=np.float32) / 255.0

def edge_map(img):
    g = gray_array(img)
    med = float(np.median(g))
    return cv2.Canny(
        g,
        int(max(0, 0.66 * med)),
        int(min(255, 1.33 * med)),
    ) > 0

def edge_f1(ref, cand, tolerance_px=1):
    r = edge_map(ref)
    c = edge_map(cand.resize(ref.size, Image.Resampling.BILINEAR))
    k = np.ones((2 * tolerance_px + 1, 2 * tolerance_px + 1), np.uint8)
    rd = cv2.dilate(r.astype(np.uint8), k) > 0
    cd = cv2.dilate(c.astype(np.uint8), k) > 0
    p = np.sum(c & rd) / max(np.sum(c), 1)
    rec = np.sum(r & cd) / max(np.sum(r), 1)
    return float(2 * p * rec / (p + rec)) if p + rec else 1.0

def component_count(img):
    g = gray_array(img)
    _, bw = cv2.threshold(
        g, 0, 255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
    )
    n, _, stats, _ = cv2.connectedComponentsWithStats(bw, connectivity=8)
    return 0 if n <= 1 else int(np.sum(stats[1:, cv2.CC_STAT_AREA] >= 2))

def component_retention(ref, cand):
    a, b = component_count(ref), component_count(cand)
    return (1.0 if b == 0 else 0.0) if a == 0 else float(min(a, b) / max(a, b, 1))

def color_difference_metrics(ref, cand):
    max_side = 640
    scale = min(1.0, max_side / max(ref.size))
    size = (
        max(1, round(ref.width * scale)),
        max(1, round(ref.height * scale)),
    )
    r = rgb_float(ref, size)
    c = rgb_float(cand, size)
    de = deltaE_ciede2000(rgb2lab(r), rgb2lab(c))
    return {
        "deltae_mean": float(np.mean(de)),
        "deltae_p95": float(np.percentile(de, 95)),
    }

def evaluate(ref, cand):
    c = cand.image.resize(ref.size, Image.Resampling.BILINEAR)
    color = color_difference_metrics(ref, c)
    return {
        "ssim": float(ssim(gray_array(ref), gray_array(c), data_range=255)),
        "edge_f1": edge_f1(ref, c),
        "component_retention": component_retention(ref, c),
        **color,
    }

def passes_gate(cls, m):
    failures = []

    if cls in {"PHOTO_COLOR", "PHOTO_GRAYSCALE"}:
        if m["ssim"] < QUALITY["photo_ssim_min"]:
            failures.append("photo_ssim")

    elif cls == "LINE_ART_BILEVEL":
        if m["edge_f1"] < QUALITY["bilevel_edge_f1_min"]:
            failures.append("bilevel_edge_f1")
        if m["component_retention"] < QUALITY["component_retention_min"]:
            failures.append("component_retention")

    elif cls in {"LINE_ART_GRAYSCALE", "SCREENSHOT_OR_TEXT_GRAPHIC", "ALPHA_GRAPHIC"}:
        if m["ssim"] < QUALITY["graphic_ssim_min"]:
            failures.append("graphic_ssim")
        if m["edge_f1"] < QUALITY["edge_f1_min"]:
            failures.append("edge_f1")

        if cls in {"LINE_ART_GRAYSCALE", "SCREENSHOT_OR_TEXT_GRAPHIC"}:
            if m["component_retention"] < QUALITY["component_retention_min"]:
                failures.append("component_retention")

        if cls in {"SCREENSHOT_OR_TEXT_GRAPHIC", "ALPHA_GRAPHIC"}:
            if m["deltae_mean"] > QUALITY["color_deltae_mean_max"]:
                failures.append("deltae_mean")
            if m["deltae_p95"] > QUALITY["color_deltae_p95_max"]:
                failures.append("deltae_p95")

    else:
        if m["ssim"] < QUALITY["mixed_ssim_min"]:
            failures.append("mixed_ssim")
        if m["edge_f1"] < QUALITY["edge_f1_min"]:
            failures.append("edge_f1")
        if m["deltae_mean"] > QUALITY["color_deltae_mean_max"]:
            failures.append("deltae_mean")
        if m["deltae_p95"] > QUALITY["color_deltae_p95_max"]:
            failures.append("deltae_p95")

    return not failures, failures

def preferred_quality_zone(cls, m):
    if cls not in {
        "SCREENSHOT_OR_TEXT_GRAPHIC",
        "ALPHA_GRAPHIC",
        "MIXED_CONTENT",
        "LINE_ART_GRAYSCALE",
    }:
        return False

    if m["ssim"] < QUALITY["preferred_graphic_ssim_min"]:
        return False
    if m["edge_f1"] < QUALITY["preferred_edge_f1_min"]:
        return False

    if cls in {"SCREENSHOT_OR_TEXT_GRAPHIC", "LINE_ART_GRAYSCALE"}:
        if m["component_retention"] < QUALITY["preferred_component_retention_min"]:
            return False

    if cls in {"SCREENSHOT_OR_TEXT_GRAPHIC", "ALPHA_GRAPHIC", "MIXED_CONTENT"}:
        if m["deltae_mean"] > QUALITY["preferred_color_deltae_mean_max"]:
            return False
        if m["deltae_p95"] > QUALITY["preferred_color_deltae_p95_max"]:
            return False

    return True


## 05 — LM Studio / Gemma structured metadata

In [ ]:

LLM_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "image_semantic_metadata",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "title": {"type":"string"},
                "alt_text": {"type":"string","maxLength":MAX_ALT_TEXT_CHARS},
                "description": {"type":"string"},
                "extended_description": {"type":"string"},
                "keywords": {"type":"array","items":{"type":"string"},"maxItems":MAX_KEYWORDS},
                "semantic_visual_type": {"type":"string"},
                "contains_visible_text": {"type":"boolean"},
                "visible_text_summary": {"type":"string"},
            },
            "required": ["title","alt_text","description","extended_description","keywords","semantic_visual_type","contains_visible_text","visible_text_summary"],
            "additionalProperties": False,
        },
    },
}

def http_json(url, payload=None, timeout=20):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url,data=data,headers={"Content-Type":"application/json"},method="GET" if payload is None else "POST")
    with urllib.request.urlopen(req,timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))

def lm_studio_available():
    try:
        http_json(f"{LM_STUDIO_BASE_URL}/models",timeout=3)
        return True
    except Exception:
        return False

def resolve_lm_studio_model(preferred):
    ids=[m.get("id") for m in http_json(f"{LM_STUDIO_BASE_URL}/models",timeout=5).get("data",[]) if m.get("id")]
    if preferred in ids: return preferred
    for mid in ids:
        if preferred.lower() in mid.lower(): return mid
    for mid in ids:
        if "gemma" in mid.lower() and "12b" in mid.lower(): return mid
    raise RuntimeError(f"Gemma 12B not found. Available: {ids}")

def image_data_url(path):
    mime="image/jpeg" if path.suffix.lower() in {".jpg",".jpeg"} else "image/png"
    return f"data:{mime};base64,{base64.b64encode(path.read_bytes()).decode('ascii')}"

def generate_semantic_metadata(image_path, row):
    model_id=resolve_lm_studio_model(LM_STUDIO_MODEL)
    context={
        "section": row.get("section_heading") or "",
        "source_caption": row.get("source_caption") or "",
        "nearby_document_text": (row.get("nearby_text") or "")[:1800],
    }
    system=(
        "Erzeuge Metadaten für Bilder aus deutschen technischen Dokumenten. "
        "Beschreibe ausschließlich das tatsächlich gelieferte Bild. Nutze Dokumentkontext nur zur Plausibilisierung. "
        "Erfinde keinen unlesbaren Text und keine unsicheren Objekte. alt_text muss kurz und barrierefrei sein. "
        "description ist eine kompakte visuelle Beschreibung; extended_description darf für RAG ausführlicher sein. "
        "keywords enthält spezifische, nicht redundante Suchbegriffe."
    )
    payload={
        "model":model_id,
        "messages":[
            {"role":"system","content":system},
            {"role":"user","content":[
                {"type":"text","text":"Dokumentkontext:\n"+json.dumps(context,ensure_ascii=False,indent=2)},
                {"type":"image_url","image_url":{"url":image_data_url(image_path)}},
            ]},
        ],
        "temperature":LM_STUDIO_TEMPERATURE,
        "max_tokens":LM_STUDIO_MAX_TOKENS,
        "response_format":LLM_SCHEMA,
    }
    response=http_json(f"{LM_STUDIO_BASE_URL}/chat/completions",payload,LM_STUDIO_TIMEOUT_SECONDS)
    result=json.loads(response["choices"][0]["message"]["content"])
    result["alt_text"]=result["alt_text"].strip()[:MAX_ALT_TEXT_CHARS]
    result["keywords"]=list(dict.fromkeys(k.strip() for k in result["keywords"] if k.strip()))[:MAX_KEYWORDS]
    result["_model_id"]=model_id
    result["_generated_at"]=datetime.now(timezone.utc).isoformat()
    return result

def fallback_semantic_metadata(row):
    caption=(row.get("source_caption") or "").strip()
    section=(row.get("section_heading") or "").strip()
    return {
        "title": caption or section or f"Bild auf Seite {row.get('page_no')}",
        "alt_text": (caption or "Bild")[:MAX_ALT_TEXT_CHARS],
        "description": caption,
        "extended_description": "",
        "keywords": [],
        "semantic_visual_type": "",
        "contains_visible_text": False,
        "visible_text_summary": "",
        "_model_id": None,
        "_generated_at": datetime.now(timezone.utc).isoformat(),
    }


## 06 — XMP construction

In [ ]:

RDF_NS="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
DC_NS="http://purl.org/dc/elements/1.1/"
IPTC_CORE_NS="http://iptc.org/std/Iptc4xmpCore/1.0/xmlns/"

def lang_alt(value,lang=METADATA_LANGUAGE):
    value=escape(value or "")
    return f'<rdf:Alt><rdf:li xml:lang="x-default">{value}</rdf:li><rdf:li xml:lang="{escape(lang)}">{value}</rdf:li></rdf:Alt>'

def bag(values):
    return "<rdf:Bag>"+"".join(f"<rdf:li>{escape(v)}</rdf:li>" for v in values if v)+"</rdf:Bag>"

def build_xmp_packet(semantic, source_info, technical_class, output_width, output_height, method, model):
    title=semantic.get("title") or source_info.get("source_caption") or ""
    alt=semantic.get("alt_text") or source_info.get("source_caption") or ""
    desc=semantic.get("description") or source_info.get("source_caption") or ""
    ext=semantic.get("extended_description") or ""
    kws=semantic.get("keywords") or []
    generated=semantic.get("_generated_at") or datetime.now(timezone.utc).isoformat()

    custom={
        "schemaVersion":DOCRAG_SCHEMA_VERSION,
        "assetId":source_info.get("picture_ref") or "",
        "sourceDocument":source_info.get("source_document") or "",
        "sourcePage":str(source_info.get("page_no") or ""),
        "sourcePictureRef":source_info.get("picture_ref") or "",
        "sourceBBox":json.dumps(source_info.get("bbox"),separators=(",",":")),
        "sourceCoordOrigin":source_info.get("bbox_coord_origin") or "",
        "sourceImageUri":source_info.get("source_uri") or "",
        "sourceWidth":str(source_info.get("source_width") or ""),
        "sourceHeight":str(source_info.get("source_height") or ""),
        "sourceDpi":str(source_info.get("source_dpi") or ""),
        "sourceCaption":source_info.get("source_caption") or "",
        "sectionHeading":source_info.get("section_heading") or "",
        "optimizedWidth":str(output_width),
        "optimizedHeight":str(output_height),
        "maxPpi":str(MAX_PPI),
        "technicalClass":technical_class,
        "descriptionMethod":method,
        "descriptionModel":model or "",
        "metadataGeneratedAt":generated,
    }
    custom_xml="\n".join(f"<docrag:{k}>{escape(v)}</docrag:{k}>" for k,v in custom.items())

    packet=f"""<?xpacket begin="﻿" id="W5M0MpCehiHzreSzNTczkc9d"?>
<x:xmpmeta xmlns:x="adobe:ns:meta/">
<rdf:RDF xmlns:rdf="{RDF_NS}">
<rdf:Description rdf:about="" xmlns:dc="{DC_NS}" xmlns:Iptc4xmpCore="{IPTC_CORE_NS}" xmlns:docrag="{DOCRAG_NAMESPACE_URI}">
<dc:title>{lang_alt(title)}</dc:title>
<dc:description>{lang_alt(desc)}</dc:description>
<dc:subject>{bag(kws)}</dc:subject>
<dc:language><rdf:Bag><rdf:li>{escape(METADATA_LANGUAGE)}</rdf:li></rdf:Bag></dc:language>
<Iptc4xmpCore:AltTextAccessibility>{lang_alt(alt)}</Iptc4xmpCore:AltTextAccessibility>
<Iptc4xmpCore:ExtDescrAccessibility>{lang_alt(ext)}</Iptc4xmpCore:ExtDescrAccessibility>
{custom_xml}
</rdf:Description>
</rdf:RDF>
</x:xmpmeta>
<?xpacket end="w"?>"""
    return packet.encode("utf-8")


## 07 — Native JPEG/PNG XMP embedding

In [ ]:

PNG_SIGNATURE=b"\x89PNG\r\n\x1a\n"
PNG_XMP_KEYWORD=b"XML:com.adobe.xmp"
JPEG_XMP_HEADER=b"http://ns.adobe.com/xap/1.0/\x00"

def png_chunk(kind,payload):
    body=kind+payload
    return struct.pack(">I",len(payload))+body+struct.pack(">I",binascii.crc32(body)&0xffffffff)

def strip_png_xmp(data):
    if not data.startswith(PNG_SIGNATURE): raise ValueError("Not PNG")
    out=bytearray(PNG_SIGNATURE); pos=len(PNG_SIGNATURE)
    while pos<len(data):
        length=struct.unpack(">I",data[pos:pos+4])[0]
        kind=data[pos+4:pos+8]; payload=data[pos+8:pos+8+length]
        full=data[pos:pos+12+length]; pos+=12+length
        if not (kind==b"iTXt" and payload.startswith(PNG_XMP_KEYWORD+b"\x00")):
            out.extend(full)
        if kind==b"IEND": break
    return bytes(out)

def embed_png_xmp(data,xmp):
    data=strip_png_xmp(data)
    payload=PNG_XMP_KEYWORD+b"\x00"+b"\x00"+b"\x00"+b"\x00"+b"\x00"+xmp
    xchunk=png_chunk(b"iTXt",payload)
    out=bytearray(PNG_SIGNATURE); pos=len(PNG_SIGNATURE); inserted=False
    while pos<len(data):
        length=struct.unpack(">I",data[pos:pos+4])[0]
        kind=data[pos+4:pos+8]; full=data[pos:pos+12+length]; pos+=12+length
        if kind==b"IDAT" and not inserted:
            out.extend(xchunk); inserted=True
        out.extend(full)
        if kind==b"IEND": break
    if not inserted: raise ValueError("PNG contains no IDAT")
    return bytes(out)

def extract_png_xmp(data):
    if not data.startswith(PNG_SIGNATURE): return None
    pos=len(PNG_SIGNATURE)
    while pos<len(data):
        length=struct.unpack(">I",data[pos:pos+4])[0]
        kind=data[pos+4:pos+8]; payload=data[pos+8:pos+8+length]; pos+=12+length
        if kind==b"iTXt" and payload.startswith(PNG_XMP_KEYWORD+b"\x00"):
            rest=payload[len(PNG_XMP_KEYWORD)+1:]
            flag=rest[0]; rest=rest[2:]
            i=rest.find(b"\x00"); rest=rest[i+1:]
            j=rest.find(b"\x00"); text=rest[j+1:]
            if flag!=0: raise NotImplementedError("compressed XMP iTXt")
            return text
        if kind==b"IEND": break
    return None

def iter_jpeg_segments(data):
    if not data.startswith(b"\xff\xd8"): raise ValueError("Not JPEG")
    pos=2
    while pos<len(data) and data[pos]==0xff:
        start=pos
        while pos<len(data) and data[pos]==0xff: pos+=1
        marker=data[pos]; pos+=1
        if marker in (0xd8,0xd9) or 0xd0<=marker<=0xd7:
            yield marker,start,pos,b""; continue
        seglen=struct.unpack(">H",data[pos:pos+2])[0]
        end=pos+seglen; payload=data[pos+2:end]
        yield marker,start,end,payload
        if marker==0xda: break
        pos=end

def strip_jpeg_xmp(data):
    out=bytearray(b"\xff\xd8"); last=2
    for marker,start,end,payload in iter_jpeg_segments(data):
        if start>last: out.extend(data[last:start])
        if not (marker==0xe1 and payload.startswith(JPEG_XMP_HEADER)):
            out.extend(data[start:end])
        last=end
        if marker==0xda:
            out.extend(data[end:]); return bytes(out)
    out.extend(data[last:]); return bytes(out)

def embed_jpeg_xmp(data,xmp):
    data=strip_jpeg_xmp(data)
    payload=JPEG_XMP_HEADER+xmp
    seglen=len(payload)+2
    if seglen>65535: raise ValueError("XMP too large for single standard JPEG APP1")
    return data[:2]+b"\xff\xe1"+struct.pack(">H",seglen)+payload+data[2:]

def extract_jpeg_xmp(data):
    for marker,_,_,payload in iter_jpeg_segments(data):
        if marker==0xe1 and payload.startswith(JPEG_XMP_HEADER):
            return payload[len(JPEG_XMP_HEADER):]
    return None

def embed_xmp(data,fmt,xmp):
    return embed_png_xmp(data,xmp) if fmt=="PNG" else embed_jpeg_xmp(data,xmp)

def extract_xmp(data,fmt):
    return extract_png_xmp(data) if fmt=="PNG" else extract_jpeg_xmp(data)

def parse_xmp(xmp):
    text=xmp.decode("utf-8")
    cleaned=re.sub(r"<\?xpacket[^>]*\?>","",text).strip()
    root=ET.fromstring(cleaned)
    ns={"rdf":RDF_NS,"dc":DC_NS,"iptc":IPTC_CORE_NS}
    def alt(xpath):
        el=root.find(xpath,ns)
        if el is None: return ""
        vals=[li.text for li in el.findall(".//rdf:li",ns) if li.text]
        return vals[0] if vals else ""
    return {
        "title":alt(".//dc:title"),
        "description":alt(".//dc:description"),
        "alt_text":alt(".//iptc:AltTextAccessibility"),
        "extended_description":alt(".//iptc:ExtDescrAccessibility"),
        "keywords":[li.text or "" for li in root.findall(".//dc:subject/rdf:Bag/rdf:li",ns)],
    }

def validate_written_image(path,expected_format):
    data=path.read_bytes(); xmp=extract_xmp(data,expected_format)
    if xmp is None: raise ValueError("XMP read-back failed")
    parsed=parse_xmp(xmp)
    with Image.open(path) as im:
        im.load()
        if im.format != expected_format: raise ValueError(f"format mismatch {im.format}")
        size=im.size
    return {"size":size,"xmp_bytes":len(xmp),"metadata":parsed}


## 08 — Production processing

In [ ]:

records=[]
path_rewrites={}
llm_ready=ENABLE_LLM and lm_studio_available()

print("LLM enabled:", ENABLE_LLM)
print("LLM ready:", llm_ready)

for _, row in assets.iterrows():
    base={
        "picture_ref":row["picture_ref"],
        "source_uri":row["source_uri"],
        "page_no":row["page_no"],
        "source_caption":row["source_caption"],
        "section_heading":row["section_heading"],
    }

    if not row["exists"]:
        records.append({**base,"status":"SOURCE_MISSING"})
        continue

    src=Path(row["resolved_path"])
    source_bytes=src.stat().st_size
    feats=technical_features(src)
    cls,reasons=classify_image(feats)
    ref=resize_reference(src,int(row["target_width"]),int(row["target_height"]))

    evaluated=[]
    for cand in candidate_family(ref,cls):
        metrics=evaluate(ref,cand)
        ok,failures=passes_gate(cls,metrics)
        evaluated.append({
            "candidate": cand,
            "metrics": metrics,
            "passes": ok,
            "failures": failures,
            "preferred": ok and preferred_quality_zone(cls, metrics),
        })

    acceptable=[x for x in evaluated if x["passes"]]
    if not acceptable:
        records.append({**base,"status":"NO_CANDIDATE_PASSED","technical_class":cls})
        continue

    preferred=[x for x in acceptable if x["preferred"]]
    selection_pool=preferred if preferred else acceptable
    selected=min(
        selection_pool,
        key=lambda x: (
            x["candidate"].nbytes,
            x["metrics"]["deltae_mean"],
            -x["metrics"]["ssim"],
        ),
    )
    cand=selected["candidate"]
    metrics=selected["metrics"]
    selection_zone="preferred" if preferred else "minimum"
    if 1-cand.nbytes/source_bytes < MIN_SAVING_RATIO:
        records.append({**base,"status":"KEEP_ORIGINAL_NO_SAVING","technical_class":cls})
        continue

    ext=".jpg" if cand.format=="JPEG" else ".png"
    output_name=Path(row["source_uri"]).stem+ext
    output_path=ARTIFACT_DIR/output_name

    # Temporary files live in the SAME directory so os.replace is atomic on the same filesystem.
    llm_temp=ARTIFACT_DIR/f".{Path(output_name).stem}.llm{ext}"
    llm_temp.write_bytes(cand.data)

    if llm_ready:
        try:
            semantic=generate_semantic_metadata(llm_temp,row)
            metadata_status="LLM_GENERATED"
            method="vision-llm"
            model=semantic.get("_model_id")
        except Exception as exc:
            semantic=fallback_semantic_metadata(row)
            metadata_status=f"SEMANTIC_METADATA_PENDING:{type(exc).__name__}"
            method="docling-context-fallback"
            model=None
    else:
        semantic=fallback_semantic_metadata(row)
        metadata_status="SEMANTIC_METADATA_PENDING"
        method="docling-context-fallback"
        model=None

    source_info={
        "picture_ref":row["picture_ref"],
        "source_document":source_document_name(doc),
        "page_no":row["page_no"],
        "bbox":{"l":row["bbox_l"],"t":row["bbox_t"],"r":row["bbox_r"],"b":row["bbox_b"]},
        "bbox_coord_origin":row["bbox_coord_origin"],
        "source_uri":row["source_uri"],
        "source_width":feats["actual_width"],
        "source_height":feats["actual_height"],
        "source_dpi":row["source_declared_dpi"],
        "source_caption":row["source_caption"],
        "section_heading":row["section_heading"],
    }

    xmp=build_xmp_packet(semantic,source_info,cls,ref.width,ref.height,method,model)
    final_data=embed_xmp(cand.data,cand.format,xmp)

    temp_final=ARTIFACT_DIR/f".{output_name}.tmp"
    temp_final.write_bytes(final_data)
    validation=validate_written_image(temp_final,cand.format)
    os.replace(temp_final,output_path)
    llm_temp.unlink(missing_ok=True)

    # Preserve the path convention emitted by Docling: <BUCH>_artifacts/<file>.
    markdown_target=f"{BUCH}_artifacts/{output_name}"
    path_rewrites[row["source_uri"]]=(markdown_target,semantic.get("alt_text") or "Bild")

    records.append({
        **base,
        "status":"OPTIMIZED",
        "source_sha256":sha256_file(src),
        "source_bytes":source_bytes,
        "source_width":feats["actual_width"],
        "source_height":feats["actual_height"],
        "target_width":ref.width,
        "target_height":ref.height,
        "technical_class":cls,
        "class_reasons":reasons,
        "selected_candidate":cand.candidate_id,
        "selected_format":cand.format,
        "selected_params":cand.params,
        "selection_zone":selection_zone,
        "ssim":metrics["ssim"],
        "edge_f1":metrics["edge_f1"],
        "component_retention":metrics["component_retention"],
        "deltae_mean":metrics["deltae_mean"],
        "deltae_p95":metrics["deltae_p95"],
        "output_path":str(output_path),
        "output_bytes":output_path.stat().st_size,
        "saving_ratio":1-output_path.stat().st_size/source_bytes,
        "metadata_status":metadata_status,
        "description_method":method,
        "description_model":model,
        "title":semantic.get("title"),
        "alt_text":semantic.get("alt_text"),
        "description":semantic.get("description"),
        "extended_description":semantic.get("extended_description"),
        "keywords":semantic.get("keywords"),
        "xmp_bytes":validation["xmp_bytes"],
        "xmp_readback_valid":True,
        "old_source_path": str(src),
        "new_source_uri": markdown_target,
    })

production=pd.DataFrame(records)
display(production[["picture_ref","status","technical_class","selected_candidate","selected_format","output_bytes","saving_ratio","metadata_status"]])


## 09 — Rewrite Markdown and write audit manifests

In [ ]:

def rewrite_markdown_images(text, rewrites):
    pattern = re.compile(r"!\[(?P<alt>[^\]]*)\]\((?P<path>[^)]+)\)")
    def repl(m):
        old = m.group("path")
        if old not in rewrites:
            return m.group(0)
        new_path, new_alt = rewrites[old]
        new_alt = (
            str(new_alt)
            .replace("\\", "\\\\")
            .replace("[", "\\[")
            .replace("]", "\\]")
        )
        return f"![{new_alt}]({new_path})"
    return pattern.sub(repl, text)

def atomic_write_text(path: Path, text: str):
    tmp = path.with_name(f".{path.name}.tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def atomic_write_json(path: Path, obj: dict):
    tmp = path.with_name(f".{path.name}.tmp")
    tmp.write_text(
        json.dumps(obj, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    os.replace(tmp, path)

# 1) Markdown in memory aktualisieren.
optimized_md = rewrite_markdown_images(md_text, path_rewrites)

# 2) Docling JSON auf die neuen Asset-URIs/-Formate/-Abmessungen aktualisieren.
updated_doc = json.loads(json.dumps(doc))  # deep copy, JSON-safe
prod_by_ref = {
    row["picture_ref"]: row
    for _, row in production[production["status"] == "OPTIMIZED"].iterrows()
}

for picture in updated_doc.get("pictures", []):
    ref = picture.get("self_ref")
    if ref not in prod_by_ref:
        continue

    prow = prod_by_ref[ref]
    image = picture.setdefault("image", {})
    image["uri"] = prow["new_source_uri"]
    image["mimetype"] = (
        "image/jpeg" if prow["selected_format"] == "JPEG" else "image/png"
    )
    image["dpi"] = MAX_PPI

    size = image.setdefault("size", {})
    size["width"] = int(prow["target_width"])
    size["height"] = int(prow["target_height"])

# 3) Commit Markdown + JSON atomar pro Datei.
#    Bilddateien wurden bereits einzeln validiert und via os.replace committed.
atomic_write_text(MARKDOWN_FILE, optimized_md)
atomic_write_json(DOCLING_JSON, updated_doc)

# 4) Alte Rasterdatei nur löschen, wenn die Dateiendung/der Pfad geändert wurde
#    und das neue Ziel sicher vorhanden ist.
removed_old_assets = []
for _, row in production[production["status"] == "OPTIMIZED"].iterrows():
    old_path = Path(row["old_source_path"])
    new_path = Path(row["output_path"])

    if old_path.resolve() != new_path.resolve():
        if not new_path.exists():
            raise RuntimeError(
                f"Neues Asset fehlt; altes wird NICHT gelöscht: {new_path}"
            )
        old_path.unlink(missing_ok=True)
        removed_old_assets.append(str(old_path))

# 5) Kleine Audit-Manifeste im Dokumentordner.
def json_safe(v):
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return None if np.isnan(v) else float(v)
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, dict):
        return {str(k): json_safe(val) for k, val in v.items()}
    if isinstance(v, list):
        return [json_safe(x) for x in v]
    return v

manifest = {
    "schema_version": "production-0.6-inplace",
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_document": source_document_name(doc),
    "document_dir": str(DOCUMENT_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
    "max_ppi": MAX_PPI,
    "canonical_formats": ["JPEG", "PNG"],
    "in_place": True,
    "llm": {
        "enabled": ENABLE_LLM,
        "base_url": LM_STUDIO_BASE_URL,
        "configured_model": LM_STUDIO_MODEL,
    },
    "removed_old_assets": removed_old_assets,
    "assets": [
        json_safe(x)
        for x in production.to_dict(orient="records")
    ],
}
atomic_write_json(MANIFEST_JSON, manifest)

csv_df = production.copy()
for col in csv_df.columns:
    if csv_df[col].map(lambda x: isinstance(x, (list, dict))).any():
        csv_df[col] = csv_df[col].map(
            lambda x: json.dumps(x, ensure_ascii=False, sort_keys=True)
            if isinstance(x, (list, dict))
            else x
        )

csv_tmp = MANIFEST_CSV.with_name(f".{MANIFEST_CSV.name}.tmp")
csv_df.to_csv(csv_tmp, index=False)
os.replace(csv_tmp, MANIFEST_CSV)

print("Markdown aktualisiert :", MARKDOWN_FILE)
print("Docling JSON aktualisiert:", DOCLING_JSON)
print("Artifacts in-place    :", ARTIFACT_DIR)
print("Alte Assets entfernt  :", len(removed_old_assets))
print("Manifest JSON         :", MANIFEST_JSON)
print("Manifest CSV          :", MANIFEST_CSV)


## 10 — Final XMP read-back validation

In [ ]:

validation = []

for _, row in production[production["status"] == "OPTIMIZED"].iterrows():
    result = validate_written_image(
        Path(row["output_path"]),
        row["selected_format"],
    )
    validation.append({
        "picture_ref": row["picture_ref"],
        "format": row["selected_format"],
        "width": result["size"][0],
        "height": result["size"][1],
        "xmp_bytes": result["xmp_bytes"],
        "title": result["metadata"]["title"],
        "alt_text": result["metadata"]["alt_text"],
        "keywords_count": len(result["metadata"]["keywords"]),
    })

validation_df = pd.DataFrame(validation)
display(validation_df)

# Konsistenz über alle drei Dateien/Strukturen hinweg.
assert MARKDOWN_FILE.exists()
assert DOCLING_JSON.exists()
assert ARTIFACT_DIR.is_dir()
assert MANIFEST_JSON.exists()

for _, row in production[production["status"] == "OPTIMIZED"].iterrows():
    assert Path(row["output_path"]).exists()
    assert bool(row["xmp_readback_valid"])

# Prüfen, dass aktualisierte JSON-URIs tatsächlich auf existierende Assets zeigen.
final_doc = load_json(DOCLING_JSON)
for picture in final_doc.get("pictures", []):
    uri = picture.get("image", {}).get("uri")
    if uri:
        resolved = DOCUMENT_DIR / uri
        assert resolved.exists(), f"Broken Docling image URI: {uri}"

final_md = MARKDOWN_FILE.read_text(encoding="utf-8")
for new_path, _alt in path_rewrites.values():
    assert new_path in final_md, f"Markdown reference missing: {new_path}"

print("Final validation passed.")



## 11 — Enabling Gemma 4 12B locally

1. Load a vision-capable Gemma 4 12B model in LM Studio.
2. Start the local server from the **Developer** tab.
3. Ensure it is visible through `GET http://localhost:1234/v1/models`.
4. Set `ENABLE_LLM = True`.
5. If necessary, set `LM_STUDIO_MODEL` to the exact model identifier returned by `/v1/models`.
6. Re-run the notebook.

If LM Studio is unavailable, raster optimization still completes. Deterministic Docling provenance is embedded into XMP and the manifest records `SEMANTIC_METADATA_PENDING`. This permits a later metadata-only retry without another lossy raster encoding.

Before archive-wide deployment, replace `urn:docrag:metadata:1.0` with an organisation-owned namespace URI and independently inspect sample files with current ExifTool.



## v0.5 operational note

For best palette quality, install current `pngquant` and verify that `PNGQUANT_BINARY` resolves to an executable path. If `pngquant` is unavailable, the notebook automatically falls back to Pillow while still enforcing the 16-color minimum and CIEDE2000 color-quality gates.

`COLOR_GRAPHIC_MIN_PALETTE_COLORS = 16` is a hard project policy in this revision.
